In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_AUXILIARY_FUNCTIONS

In [0]:
spark.table('bronze.faturamento_erp_202510_202609').show()

In [0]:
spark.table('bronze.faturamento_erp_202510_202609').printSchema()

In [0]:
TABLE_ORIGIN = 'faturamento_erp_202510_202609'

TABLE_DESTINY = 'invoice_erp_202510_to_202609'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
                  .withColumnsRenamed(
                                       {
                                         'num_fatura' : 'invoice_id',
                                         'id_cliente' : 'customer_id',
                                         'mes_referencia' : 'month',
                                         'data_emissao' : 'invoice_date',
                                         'kwh_faturado' : 'kwh_invoiced',
                                         'tarifa_cheia_aplicada' : 'full_charge_applied',
                                         'desconto_aplicado' : 'discount_applied',
                                         'valor_fio_b_deduzido' : 'wire_b_deducted',
                                         'valor_faturado' : 'invoiced_value',
                                       }
                                     )
                  .withColumn('kwh_invoiced', F.col('kwh_invoiced').cast('bigint'))
                  .withColumn('full_charge_applied', F.col('full_charge_applied').cast('float'))
                  .withColumn('discount_applied', F.col('discount_applied').cast('float'))
                  .withColumn('wire_b_deducted', F.col('wire_b_deducted').cast('float'))
                  .withColumn('invoiced_value', F.col('invoiced_value').cast('float'))
     )

In [0]:
# Incremental load: never replaces silver.invoice_erp_202510_to_202609 — appends only new invoices
target = f"silver.{TABLE_DESTINY}"
key_cols = ['invoice_id']

if not spark.catalog.tableExists(target):
    # first load: table does not exist yet — create it
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    print(f"✓ {target} created ({df.count()} rows)")
else:
    # append only records whose key is not present in silver yet
    df_new = df.join(spark.table(target).select(*key_cols).distinct(), on=key_cols, how='left_anti')
    n_new = df_new.count()
    if n_new:
        df_new.write.format("delta").mode("append").saveAsTable(target)
        print(f"✓ {target} appended {n_new} new row(s)")
    else:
        print(f"↷ {target}: no new records — nothing appended")